# How to locate a buggy sub with subwise profiling

## Why use subwise profiling

When a large FTQC circuit uses more T gates, more ancillas, or more Toffolis than you expected, the aggregate count from `GateCountEvaluatorHooks` tells you *that* something is off but not *where*. The usual workaround is to chop the circuit into pieces by hand, compile each piece, and count each piece in isolation until the offending sub shows up.

Subwise profiling removes that detour: it walks the compiled `MachineSub` once and reports, per `Sub`, its call count, qubit usage, and primitive-gate breakdown. The sub whose numbers do not match your expectation is the sub to inspect.

## Build a parametric circuit with nested subs

Suppose you are writing your own decomposition of a multi-controlled X, `MCX(n)`, perhaps because your target gateset or ancilla budget rules out `std.MultiControlled` directly. You expect the Toffoli count to grow **linearly** in the number of controls `n`: roughly one Toffoli per added control.

We will build `MCX(n)` recursively: each level peels off one control and recurses on the rest. The **intended** body has one recursive call per level. The **buggy** body has a copy-paste that recurses twice. (The decomposition below is illustrative, not a correct MCX unitary; the point is gate-count debugging.)


In [ ]:
from collections.abc import Sequence

from quri_parts.qsub.opsub import ParamUnitarySubDef, param_opsub
from quri_parts.qsub.register import QRegSpec
from quri_parts.qsub.sub import SubBuilder
from quri_parts.qsub.lib import std


# Someone defined MCX by hand, but incorrect
class _MCX(ParamUnitarySubDef[int]):
    name = "MCX"

    def qubit_count_fn(self, n: int) -> int:
        return n + 1

    def qregs_fn(self, n: int) -> Sequence[QRegSpec]:
        return (QRegSpec("ctrl", n), QRegSpec("tgt", 1))

    def sub(self, b: SubBuilder, n: int) -> None:
        ctrls = b.qregs["ctrl"].qubits
        (tgt,) = b.qregs["tgt"].qubits
        if n == 1:
            b.add_op(std.CNOT, (ctrls[0], tgt))
            return
        if n == 2:
            b.add_op(std.Toffoli, (ctrls[0], ctrls[1], tgt))
            return
        # Bug: recursive call pasted twice -> Toffoli(n) = 2*Toffoli(n-1) + 1.
        b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))
        b.add_op(std.Toffoli, (ctrls[-2], ctrls[-1], tgt))
        b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))


MCX, _ = param_opsub(_MCX)


## Step 1. First approach: analysis with aggregate counting

Sweep `n` and run the existing `GateCountEvaluatorHooks`. The Toffoli count should grow by one per added control. Instead it doubles.


In [ ]:
from quri_parts.qsub.compile import compile
from quri_parts.qsub.evaluate import Evaluator
from quri_parts.qsub.eval import GateCountEvaluatorHooks

primitives = (
    std.X, std.H, std.T, std.Tdag, std.S, std.Sdag,
    std.CNOT, std.Toffoli,
)

for n in range(2, 8):
    compiled = compile(MCX(n), primitives)
    totals = Evaluator(GateCountEvaluatorHooks((std.Toffoli,))).run(compiled)
    print(f"n={n}  ->", {k[1]: v for k, v in totals.items()})

n=2  -> {'Toffoli': 1}
n=3  -> {'Toffoli': 3}
n=4  -> {'Toffoli': 7}
n=5  -> {'Toffoli': 15}
n=6  -> {'Toffoli': 31}
n=7  -> {'Toffoli': 63}


Expected: `Toffoli = 1, 2, 3, 4, 5, 6`. Observed: `1, 3, 7, 15, 31, 63` — that is `2^{n-1} - 1`. Aggregate alone cannot say which level is doubling.

## Step 2. Second approach with subwise profiling

`SubwiseProfileEvaluatorHooks` is the new hook. Its API mirrors `GateCountEvaluatorHooks` so it slots into the same `Evaluator` pipeline:

- Constructor takes the gates you want broken down. Pass nothing to count *every* primitive that appears.
- `run(compiled)` returns a list of entries, one per distinct sub, parameters included — `MCX(5)` and `MCX(6)` get separate entries.
- Each entry exposes:
  - `call_count` — how many times this sub is invoked across the program.
  - `qubit_count_info` — input/output width (`data`) and auxiliary qubits allocated inside the sub (`auxiliary`), plus their sum (`total`).
  - `gate_count_info` — primitive-gate breakdown, structured as scope × aggregation:

| Scope | Per single call | Across all `call_count` invocations |
|---|---|---|
| Gates written directly in this sub's body | `self_per_call` | `self` |
| Gates contributed by every sub this one calls into, summed transitively | `descendants_per_call` | `descendants` |
| Sum of the two rows above | `total_per_call` | `total` |

Flatten the nested entries with `pd.json_normalize`, keeping the columns the analysis needs (`desc` abbreviates `descendants`):

In [ ]:
import pandas as pd
from dataclasses import asdict

from quri_parts.qsub.eval import SubwiseProfileEntry, SubwiseProfileEvaluatorHooks

compiled = compile(MCX(6), primitives)
report_data = Evaluator(SubwiseProfileEvaluatorHooks(gates=(std.Toffoli,))).run(compiled)


def profile_table(report: Sequence[SubwiseProfileEntry]) -> pd.DataFrame:
    df = pd.json_normalize([asdict(e) for e in report]).fillna(0).convert_dtypes()
    df = df.rename(columns={
        "qubit_count_info.total": "qubits",
        "gate_count_info.self_per_call.Toffoli": "Toffoli self/call",
        "gate_count_info.descendants_per_call.Toffoli": "Toffoli desc/call",
        "gate_count_info.total_per_call.Toffoli": "Toffoli total/call",
        "gate_count_info.total.Toffoli": "Toffoli total",
    })
    return df[["name", "call_count", "qubits", "Toffoli self/call",
               "Toffoli desc/call", "Toffoli total/call", "Toffoli total"]]


profile_table(report_data)

,name,call_count,qubits,Toffoli self/call,Toffoli desc/call,Toffoli total/call,Toffoli total
0,MCX(6),1,7,1,30,31,31
1,MCX(5),2,6,1,14,15,30
2,MCX(4),4,5,1,6,7,28
3,MCX(3),8,4,1,2,3,24
4,MCX(2),16,3,1,0,1,16


Two things jump out from the table above:

1. `Toffoli self/call` is `1` at every level. No single body inlines 'too many' Toffolis.
2. `call_count` doubles per level (`1, 2, 4, 8, 16`) and `Toffoli desc/call` grows by roughly 2x per level (`30, 14, 6, 2, 0`). The blow-up lives in the **child fan-out**, not in any one body.


## Step 3. Zoom in on the offending sub

Pick a single entry out of the report and inspect it on its own — here we filter by `name`.

In [ ]:
mcx5_entry = next(e for e in report_data if e.name == "MCX(5)")
print("name:", mcx5_entry.name)
print("calls:", mcx5_entry.call_count)
print("qubits:", mcx5_entry.qubit_count_info)
print("self (per call):", mcx5_entry.gate_count_info["self_per_call"])
print("descendants (per call):", mcx5_entry.gate_count_info["descendants_per_call"])
print("total (per call):", mcx5_entry.gate_count_info["total_per_call"])

name: MCX(5)
calls: 2
qubits: {'data': 6, 'auxiliary': 0, 'total': 6}
self (per call): {'Toffoli': 1}
descendants (per call): {'Toffoli': 14}
total (per call): {'Toffoli': 15}


`MCX(5)`: `self_per_call = {Toffoli: 1}`, `total_per_call = 15`. The gap is below it, not in this body.

## Step 4. Fix and re-check

Delete the duplicate recursive call from `_MCX.sub`, redefine the sub, and sweep `n` again. The Toffoli count should now grow by one per added control.


In [ ]:
class _MCX(ParamUnitarySubDef[int]):  # noqa: F811 — replacing the buggy definition
    name = "MCX"

    def qubit_count_fn(self, n: int) -> int:
        return n + 1

    def qregs_fn(self, n: int) -> Sequence[QRegSpec]:
        return (QRegSpec("ctrl", n), QRegSpec("tgt", 1))

    def sub(self, b: SubBuilder, n: int) -> None:
        ctrls = b.qregs["ctrl"].qubits
        (tgt,) = b.qregs["tgt"].qubits
        if n == 1:
            b.add_op(std.CNOT, (ctrls[0], tgt))
            return
        if n == 2:
            b.add_op(std.Toffoli, (ctrls[0], ctrls[1], tgt))
            return
        b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))
        b.add_op(std.Toffoli, (ctrls[-2], ctrls[-1], tgt))


MCX, _ = param_opsub(_MCX)

for n in range(2, 8):
    compiled = compile(MCX(n), primitives)
    totals = Evaluator(GateCountEvaluatorHooks((std.Toffoli,))).run(compiled)
    print(f"n={n}  ->", {k[1]: v for k, v in totals.items()})

n=2  -> {'Toffoli': 1}
n=3  -> {'Toffoli': 2}
n=4  -> {'Toffoli': 3}
n=5  -> {'Toffoli': 4}
n=6  -> {'Toffoli': 5}
n=7  -> {'Toffoli': 6}


Subwise profiling on the fixed version should now show `call_count = 1` for every level and `Toffoli desc/call` shrinking by one per level.

In [ ]:
compiled = compile(MCX(6), primitives)
report_data_fixed = Evaluator(SubwiseProfileEvaluatorHooks(gates=(std.Toffoli,))).run(compiled)

profile_table(report_data_fixed)

,name,call_count,qubits,Toffoli self/call,Toffoli desc/call,Toffoli total/call,Toffoli total
0,MCX(6),1,7,1,4,5,5
1,MCX(5),1,6,1,3,4,4
2,MCX(4),1,5,1,2,3,3
3,MCX(3),1,4,1,1,2,2
4,MCX(2),1,3,1,0,1,1


In production, prefer audited primitives like `std.MultiControlled`; subwise profiling is for the cases where you have to roll your own sub.